<a href="https://colab.research.google.com/github/charlesbihdev/charlesbihdev-ML-DataScience-Notebooks/blob/face-detection-model-testing/face_detection_model_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install facenet-pytorch pillow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.6/755.6 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2

In [1]:
import os
import sqlite3
import numpy as np
from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1
import torch

# ✅ Initialize models
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(image_size=160).to(device)
model = InceptionResnetV1(pretrained='vggface2').eval().to(device)

# ✅ Path to test folder
test_folder = "/content/drive/MyDrive/ml/my-images/test"




  0%|          | 0.00/107M [00:00<?, ?B/s]

In [4]:
# ✅ Load stored embedding from SQLite
def load_stored_embedding():
    conn = sqlite3.connect('/content/drive/MyDrive/ml/my-images/face_data.db')
    cursor = conn.cursor()
    cursor.execute("SELECT name, embedding FROM faces")
    row = cursor.fetchone()
    conn.close()
    if row:
        name, blob = row
        embedding = np.frombuffer(blob, dtype=np.float32)
        return name, embedding
    else:
        return None, None
print(load_stored_embedding())

('Charles', array([ 1.67659931e-02,  3.28802802e-02,  2.35756803e-02, -8.36805440e-03,
       -5.08446805e-02, -8.53697211e-02, -2.65511274e-02, -1.04329793e-03,
       -1.83907151e-02, -4.66327593e-02,  2.70071588e-02, -6.93027303e-02,
       -5.98627701e-02, -8.81593116e-03,  6.38301373e-02,  8.41243367e-04,
       -2.03935839e-02, -4.74914238e-02, -6.46872371e-02, -1.50104184e-02,
        6.01057522e-02, -5.99410906e-02, -2.60090120e-02,  5.91408126e-02,
        3.40993894e-04,  2.37722341e-02, -9.01128072e-03, -5.65948151e-03,
        2.84245368e-02, -1.69696733e-02, -3.44562717e-02,  4.08452041e-02,
        1.36581408e-02, -3.34928110e-02, -1.79387517e-02,  4.62631620e-02,
       -7.56968111e-02, -2.11603008e-02,  3.32606733e-02, -1.98075976e-02,
       -2.29611285e-02, -1.46825071e-02, -3.76492180e-02, -5.27740829e-02,
        7.38426894e-02,  6.66290801e-03, -3.01357359e-02, -6.66163489e-03,
        1.03649097e-02,  7.36405849e-02,  6.08424954e-02, -1.10100582e-02,
        1.317

In [6]:
# ✅ Cosine similarity function
def cosine_similarity(a, b):
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    return np.dot(a, b)

In [8]:
# ✅ Load stored face
stored_name, stored_embedding = load_stored_embedding()
if stored_embedding is None:
    print("❌ No face embedding found in DB.")
else:
    print(f"✅ Loaded embedding for: {stored_name}\n")

    # ✅ Loop through test images
    for filename in sorted(os.listdir(test_folder)):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            path = os.path.join(test_folder, filename)
            img = Image.open(path).convert('RGB')
            face = mtcnn(img)

            if face is not None:
                with torch.no_grad():
                    test_emb = model(face.unsqueeze(0).to(device)).cpu().numpy()[0]
                    score = cosine_similarity(stored_embedding, test_emb)
                    print(f"score - {score}")
                    percentage = round(score * 100, 2)
                    print(f"{filename}: 🔍 {percentage}% match with {stored_name}")
            else:
                print(f"{filename}: ❌ No face detected")


✅ Loaded embedding for: Charles

score - 0.4898799657821655
test1.jpg: 🔍 48.99% match with Charles
score - 0.8008549213409424
test2.jpg: 🔍 80.09% match with Charles
score - 0.36666351556777954
test3.jpg: 🔍 36.67% match with Charles
score - 0.2969592809677124
test4.jpg: 🔍 29.7% match with Charles
score - 0.9085537195205688
test5.jpg: 🔍 90.86% match with Charles
